# 🎯 Objetivo del taller

Entrenar un modelo de regresión lineal utilizando el algoritmo integrado Linear Learner de Amazon SageMaker, desde Google Colab, accediendo a los servicios de AWS mediante credenciales IAM.

# 🧩 1️⃣ Prerrequisitos

Cuenta de AWS activa.

Usuario IAM con permisos para usar SageMaker y S3 (política AmazonSageMakerFullAccess).

Un bucket S3 creado para almacenar datasets y resultados.
Ejemplo:

mi-bucket-ulsa-008548


Archivo CSV previamente subido a S3, por ejemplo:

s3://mi-bucket-ulsa-008548/datasets/regresion-lineal/dataset_sintetico_2GB.csv


Formato CSV sin encabezado.

Primera columna = etiqueta y.

Resto de columnas = features numéricas.

Rol IAM de ejecución para SageMaker



# Instalar dependencias

In [ ]:
!pip install --quiet boto3 sagemaker

In [ ]:
import boto3
import sagemaker
from sagemaker import image_uris

🔑 Configurar credenciales AWS (usando Colab Secrets)

En el menú de Colab → Tools > Secrets.

Agrega cuatro secretos:

AWS_ACCESS_KEY_ID

AWS_SECRET_ACCESS_KEY

AWS_SESSION_TOKEN

SAGEMAKER_ROLE


Luego, ejecuta la celda:

In [ ]:
from google.colab import userdata
import os

# Get AWS credentials from Colab Secrets
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_SESSION_TOKEN'] = userdata.get('AWS_SESSION_TOKEN')
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1' # Add your desired region here

print("AWS credentials and region set as environment variables.")

# Configuracion de SageMaker

In [ ]:
# Configuración
region = 'us-east-1'
#role = sagemaker.get_execution_role()
import boto3
boto_sess = boto3.Session(region_name=region)
sess = sagemaker.Session(boto_session=boto_sess)
bucket_name = "mi-bucket-ulsa-008548"
prefix = "datasets/regresion-lineal/"
filename="dataset_sintetico_2GB.csv"

In [ ]:
role =  userdata.get('SAGEMAKER_ROLE')

# 🧠 4️⃣ Seleccionar contenedor del algoritmo Linear Learner

In [ ]:
container = image_uris.retrieve("linear-learner", region)


# 🧮 5️⃣ Definir el Estimator

In [ ]:

# Definición del Estimator
linear = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.4xlarge",
    output_path=f"s3://{bucket_name}/output/",
    sagemaker_session=sess,
)

# Hiperparámetros
linear.set_hyperparameters(
    predictor_type="regressor",
    mini_batch_size=5000,
    epochs=5
)

# Definir dataset de entrada (en formato CSV)
train_input = sagemaker.inputs.TrainingInput(
    s3_data=f"s3://{bucket_name}/{prefix}{filename}",
    content_type="text/csv",
    input_mode="Pipe"  # streaming directo
)



In [ ]:
# Entrenar
linear.fit({"train": train_input})

In [ ]:
model_artifact = linear.model_data
print(model_artifact)